# 05 — Final Four-Model Comparison

This notebook loads **measured** comparison rows generated by the four model notebooks. It contains no hard-coded performance results.

In [ ]:

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:

from pathlib import Path

def locate_project_root():
    """Find the repository root by requiring splits/ and config/ to exist."""
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path('/content/brain_tumour_model_comparison'),
        Path('/content/brain-tumor-model-comparison'),
    ])
    content = Path('/content')
    if content.exists():
        candidates.extend([p for p in content.iterdir() if p.is_dir()])

    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except Exception:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / 'splits' / 'train.csv').is_file()
            and (candidate / 'splits' / 'val.csv').is_file()
            and (candidate / 'splits' / 'test.csv').is_file()
            and (candidate / 'config' / 'class_to_index.json').is_file()
        ):
            return candidate
    raise FileNotFoundError(
        'Could not locate the project root. Clone/upload the repository so that '
        'splits/ and config/ are available, then run the notebook again.'
    )

PROJECT_ROOT = locate_project_root()
SPLITS_DIR = PROJECT_ROOT / 'splits'
CONFIG_DIR = PROJECT_ROOT / 'config'
print('Project root:', PROJECT_ROOT)

RESULTS_DIR = PROJECT_ROOT / 'results'
COMPARISON_DIR = RESULTS_DIR / 'comparison'
PLOTS_DIR = COMPARISON_DIR / 'plots'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

ROW_FILES = {
    'Custom CNN': RESULTS_DIR/'cnn'/'phase5_finalization'/'cnn_comparison_row.csv',
    'VGG16': RESULTS_DIR/'vgg16'/'phase5_finalization'/'vgg16_comparison_row.csv',
    'ResNet50': RESULTS_DIR/'resnet50'/'phase5_finalization'/'resnet50_comparison_row.csv',
    'DenseNet121': RESULTS_DIR/'densenet121'/'phase5_finalization'/'densenet121_comparison_row.csv',
}
REQUIRED_COLUMNS = [
    'model','experiment_id','image_size','batch_size','optimizer','learning_rate',
    'max_epochs','actual_epochs','best_epoch','best_val_accuracy','test_accuracy',
    'macro_precision','macro_recall','macro_f1','roc_auc_macro','training_time_seconds',
    'inference_time','inference_time_unit','total_parameters','trainable_parameters',
    'model_size_mb','seed','gpu','dataset_version','test_samples',
    'patient_level_independence_verified'
]
missing=[str(p) for p in ROW_FILES.values() if not p.is_file()]
if missing:
    raise FileNotFoundError('Run all four model notebooks first. Missing measured comparison rows:\n'+'\n'.join(missing))

rows=[]
for label,path in ROW_FILES.items():
    df=pd.read_csv(path)
    if len(df)!=1: raise ValueError(f'{path} must contain exactly one row')
    miss=[c for c in REQUIRED_COLUMNS if c not in df.columns]
    if miss: raise ValueError(f'{path} missing columns: {miss}')
    row=df.iloc[0].copy()
    for metric in ['best_val_accuracy','test_accuracy','macro_f1','roc_auc_macro','training_time_seconds','inference_time','total_parameters']:
        if pd.isna(row[metric]): raise ValueError(f'{path}: {metric} is NaN; notebook was not fully executed')
    rows.append(row)

comparison=pd.DataFrame(rows)
paradigm={'Custom CNN':'From Scratch','VGG16':'Transfer Learning','ResNet50':'Transfer Learning','DenseNet121':'Transfer Learning'}
comparison['architecture_paradigm']=comparison['model'].map(paradigm).fillna('Unknown')
comparison['non_trainable_parameters']=comparison['total_parameters']-comparison['trainable_parameters']
comparison['inference_throughput_ips']=np.where(
    comparison['inference_time_unit'].eq('milliseconds_per_image'),
    1000.0/comparison['inference_time'], np.nan
)
comparison['validation_test_gap']=comparison['best_val_accuracy']-comparison['test_accuracy']
display(comparison)


In [ ]:

master_path=COMPARISON_DIR/'MASTER_COMPARISON_TABLE.csv'
comparison.to_csv(master_path,index=False)
print('Saved:', master_path)


In [ ]:

def save_bar(column, ylabel, title, filename):
    plt.figure(figsize=(8,5))
    plt.bar(comparison['model'], comparison[column])
    plt.ylabel(ylabel); plt.title(title); plt.xticks(rotation=20, ha='right'); plt.tight_layout()
    path=PLOTS_DIR/filename
    plt.savefig(path,dpi=180,bbox_inches='tight'); plt.show(); print('Saved:',path)

save_bar('test_accuracy','Test accuracy','Held-out Test Accuracy','test_accuracy.png')
save_bar('macro_f1','Macro F1','Held-out Test Macro F1','macro_f1.png')
save_bar('roc_auc_macro','Macro ROC-AUC','Held-out Test Macro ROC-AUC','roc_auc_macro.png')
save_bar('total_parameters','Parameters','Model Complexity','total_parameters.png')
save_bar('inference_time','Milliseconds / image','Standardized Model-only Inference Time','inference_time.png')
save_bar('training_time_seconds','Seconds','Training Time','training_time.png')


In [ ]:

# Evidence-only synthesis: no fabricated metrics and no unsupported claims.
lines=['# Final Four-Model Experimental Synthesis','',
       'All values below were loaded from measured comparison rows generated by the model notebooks.','',
       '## Master results','', comparison.to_markdown(index=False), '',
       '## Methodological limitations','',
       '- Custom CNN was trained from scratch; VGG16, ResNet50 and DenseNet121 used ImageNet pretraining, so this is a practical training-approach comparison rather than architecture-only isolation.',
       '- Patient-level independence could not be verified because reliable patient identifiers were unavailable.',
       '- Results apply to this frozen dataset/split and do not establish clinical performance across other scanners, hospitals or populations.',
       '- The held-out test set must not be used for further model tuning after these results are observed.',
]
report_path=COMPARISON_DIR/'FINAL_SYNTHESIS_REPORT.md'
report_path.write_text('\n'.join(lines),encoding='utf-8')
print('Saved:', report_path)
